# 03 · Parametrization

**Focus:** testing many edge cases without duplicating code.

Copy-pasting a test five times with different inputs is a smell: it's noisy, and when the function
changes you have to fix five things. `@pytest.mark.parametrize` lets you supply a **table of inputs
and expected outputs**; pytest generates one independent test per row. Each row passes or fails on
its own and shows up separately in the report.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### The code under test
A password-strength validator with several rules.

In [2]:
%%writefile nb03_password.py
import re

def is_valid_password(pw: str) -> bool:
    '''Valid if: >= 8 chars, has a digit, has an uppercase letter.'''
    if len(pw) < 8:
        return False
    if not re.search(r"\d", pw):
        return False
    if not re.search(r"[A-Z]", pw):
        return False
    return True

Writing nb03_password.py


### One test, a matrix of cases
Each tuple is `(password, expected)`. The `ids=` list gives each case a readable name in the output.

In [3]:
%%writefile test_nb03.py
import pytest
from nb03_password import is_valid_password

cases = [
    ("Abcdef12", True),    # meets every rule
    ("Ab1",      False),   # too short
    ("abcdefg1", False),   # no uppercase
    ("Abcdefgh", False),   # no digit
    ("Password123", True), # long, mixed, digit
    ("",         False),   # empty
]

@pytest.mark.parametrize("password, expected", cases,
                         ids=["strong", "too_short", "no_upper", "no_digit", "long_ok", "empty"])
def test_password_rules(password, expected):
    assert is_valid_password(password) == expected

Writing test_nb03.py


### Run it
Six rows → six separate tests. Look how each `id` appears as `test_password_rules[strong]`, etc.
If one row broke, you'd know *exactly* which input failed without touching the others.

In [4]:
!pytest -v test_nb03.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 6 items                                                              

test_nb03.py::test_password_rules[strong] PASSED                         [ 16%]
test_nb03.py::test_password_rules[too_short] PASSED                      [ 33%]
test_nb03.py::test_password_rules[no_upper] PASSED                       [ 50%]
test_nb03.py::test_password_rules[no_digit] PASSED       



============================== 6 passed in 0.02s ===============================


### ⚠️ Common pitfall — stacked parametrize is a *product*, not a *zip*
Two `@pytest.mark.parametrize` decorators on one test don't pair up element-by-element — they form
the **Cartesian product**. Three `x` values × two `y` values = **six** tests, not three. Handy for
grids, but easy to accidentally explode into hundreds of cases.

### 🔬 Try it yourself
**Predict:** how many tests does the stacked-decorator function below generate? Run it and count the
`[x-y]` combinations in the report.

In [5]:
%%writefile test_nb03_tryit.py
import pytest

@pytest.mark.parametrize("x", [1, 2, 3])
@pytest.mark.parametrize("y", [10, 20])
def test_grid(x, y):
    # Every (x, y) pair is generated. How many is that?
    assert x < y

Writing test_nb03_tryit.py


In [6]:
!pytest -v test_nb03_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 6 items                                                              

test_nb03_tryit.py::test_grid[10-1] PASSED                               [ 16%]
test_nb03_tryit.py::test_grid[10-2] PASSED                               [ 33%]
test_nb03_tryit.py::test_grid[10-3] PASSED                               [ 50%]
test_nb03_tryit.py::test_grid[20-1] PASSED               

### What you learned
- `@pytest.mark.parametrize("names", rows)` turns one function into many tests.
- Use `ids=[...]` to give each case a human-readable label in the report.
- Adding a new edge case = adding one row, not copy-pasting a function.

Next up: **mocking** — isolating your code from the network and disk.